# Pollen Prediction from CALIPSO / MPL Lidar Optical Properties

**Goal:** Predict ground-observed pollen concentrations (by species) using satellite/ground-based
lidar-derived aerosol optical properties (CALIPSO, MPLNET) combined with meteorological and
land-cover covariates, via a Random Forest regression model.

## Structure
1. Imports
2. Station metadata (lat/lon for each monitoring site)
3. Load per-site predictor (lidar) and target (pollen) data — generalized loop, not one block per city
4. Load and attach meteorological/land-cover covariates (CDD, day length, land-cover fractions)
5. Combine all sites into one training table
6. Train/test split
7. Train Random Forest
8. Evaluate + feature importance
9. Per-species performance metrics


## 1. Imports

In [ ]:
import os
import glob
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns

from scipy.spatial.distance import cdist

from netCDF4 import Dataset

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


## 2. Station metadata


Silver Spring (`SP`) is the only site with a co-located ground-based MPLNET lidar (`mpl_*`
variables) in addition to CALIPSO satellite lidar (`cal_*`); all other sites have CALIPSO only.

In [ ]:
STATIONS = {
    "SP":  {"name": "Silver Spring",     "lat": 39.0,  "lon": -76.9,  "has_mpl": True},
    "AM":  {"name": "Armonk",            "lat": 41.1,  "lon": -73.7},
    "AS":  {"name": "Asheville",         "lat": 35.6,  "lon": -82.6},
    "AT":  {"name": "Atlanta",           "lat": 33.8,  "lon": -84.4},
    "BK":  {"name": "Berkeley",          "lat": 38.7,  "lon": -90.3},
    "CS":  {"name": "College Station",   "lat": 30.5,  "lon": -96.3},
    "CO":  {"name": "Colorado Springs",  "lat": 38.9,  "lon": -104.8},
    "DT":  {"name": "Dayton",            "lat": 39.8,  "lon": -84.2},
    "DP":  {"name": "Draper",            "lat": 40.5,  "lon": -111.9},
    "ER":  {"name": "Erie",              "lat": 42.1,  "lon": -80.2},
    "EG":  {"name": "Eugene",            "lat": 44.0,  "lon": -123.1},
    "FM":  {"name": "Flower Mound",      "lat": 33.0,  "lon": -97.1},
    "GT":  {"name": "Georgetown",        "lat": 30.6,  "lon": -97.7},
    "GV":  {"name": "Greenville",        "lat": 34.8,  "lon": -82.4},
    "HS":  {"name": "Houston",           "lat": 29.7,  "lon": -95.4},
    "KC":  {"name": "Kansas City",       "lat": 39.1,  "lon": -94.6},
    "KV":  {"name": "Knoxville",         "lat": 35.9,  "lon": -84.0},
    "LC":  {"name": "La Crosse",         "lat": 43.9,  "lon": -91.2},
    "LV":  {"name": "Las Vegas",         "lat": 36.2,  "lon": -115.1},
    "LR":  {"name": "Little Rock",       "lat": 34.8,  "lon": -92.3},
    "MS":  {"name": "Madison",           "lat": 43.1,  "lon": -89.4},
    "MR":  {"name": "Marietta",          "lat": 34.0,  "lon": -84.5},
    "MV":  {"name": "Monroeville",       "lat": 34.4,  "lon": -80.0},
    "ML":  {"name": "Mount Laurel",      "lat": 39.9,  "lon": -74.9},
    "NY":  {"name": "New York",          "lat": 40.8,  "lon": -74.0},
    "OK1": {"name": "Oklahoma1",         "lat": 35.6,  "lon": -97.6},
    "OK2": {"name": "Oklahoma2",         "lat": 35.6,  "lon": -97.4},
    "PA":  {"name": "Papillion",         "lat": 41.1,  "lon": -95.9},
    "PD":  {"name": "Philadelphia",      "lat": 39.9,  "lon": -75.2},
    "RC":  {"name": "Rochester",         "lat": 43.1,  "lon": -77.6},
    "SA1": {"name": "San Antonio1",      "lat": 29.4,  "lon": -98.5},
    "SA2": {"name": "San Antonio2",      "lat": 29.4,  "lon": -98.5},
    "SJ":  {"name": "San Jose",          "lat": 37.3,  "lon": -122.0},
    "SS":  {"name": "Sarasota",          "lat": 27.31, "lon": -82.37},
    "ST":  {"name": "Seattle",           "lat": 47.7,  "lon": -122.3},
    "TS":  {"name": "Tulsa",             "lat": 36.1,  "lon": -95.9},
    "WC1": {"name": "Waco1",             "lat": 31.5,  "lon": -97.2},
    "WC2": {"name": "Waco2",             "lat": 31.5,  "lon": -97.2},
    "WB":  {"name": "Waterbury",         "lat": 41.6,  "lon": -73.0},
    "YK":  {"name": "York",              "lat": 39.9,  "lon": -76.7},
}

CALIPSO_DIR = "/glade/derecho/scratch/yingxiao/FROM_CHEYENNE/CALIPSO/"


## 3. Load per-site predictor (lidar) and target (pollen) data


In [ ]:
def load_site_netcdf(code, filepath):
    """Read one site's NetCDF file and return correctly-named predictor/target arrays.

    Returns a dict with keys:
        X_cal : CALIPSO-derived predictors (lidar optical properties + Month/Day/Year)
        y_cal : pollen counts collocated with CALIPSO overpasses (target)
        X_mpl : MPLNET-derived predictors (Silver Spring only)
        y_mpl : pollen counts collocated with MPLNET (Silver Spring only, target)
    """
    with Dataset(filepath, "r") as nc_file:
        out = {
            # NOTE: raw NetCDF variable 'cal_label' actually holds lidar optical properties (predictors)
            "X_cal": nc_file.variables["cal_label"][:],
            # NOTE: raw NetCDF variable 'cal_feature' actually holds pollen counts (target)
            "y_cal": nc_file.variables["cal_feature"][:],
        }
        if "mpl_label" in nc_file.variables:
            out["X_mpl"] = nc_file.variables["mpl_label"][:]
            out["y_mpl"] = nc_file.variables["mpl_feature"][:]
    return out


site_data = {}
for code_, meta in STATIONS.items():
    fname = f"ML_data_{meta['name'].replace(' ', '_')}_v2.nc"
    fpath = os.path.join(CALIPSO_DIR, fname)
    if not os.path.exists(fpath):
        print(f"WARNING: file not found for {code_} ({meta['name']}): {fpath}")
        continue
    site_data[code_] = load_site_netcdf(code_, fpath)

print(f"Loaded {len(site_data)} / {len(STATIONS)} sites.")


## 4. Pollen column names / target label list


In [ ]:
pollen_ref = pd.read_excel(
    "/glade/derecho/scratch/yingxiao/FROM_CHEYENNE/Pollen_obs/Tulsa2_forML.xlsx"
)

drop_cols = [
    "Other Grass Pollen", "Other Tree Pollen", "Other Weed Pollen",
    "Unidentified Pollen", "Date",
]
pollen_ref = pollen_ref.drop(columns=[c for c in drop_cols if c in pollen_ref.columns])

y_columns = list(pollen_ref.columns)   # e.g. ['Month','Day','Year','Total', species..., ...]
print("Target (pollen) columns:", y_columns)


## 5. Meteorological & land-cover covariates

- **CDD** — cumulative degree-days
- **DBL, ENL, C3, C4** — Deciduous Broadlead / evergreen needleleaf / C3-grass / C4-grass land-cover fractions (static per
  site, from CLM45)

It is not included here, but for the Zhang et al., 2026 paper, we also load MERRA-2 dust concentration, used later as a data-quality filter (excluding
high-dust days that can contaminate lidar-derived pollen signal).

In [ ]:
# --- CPC tmax/tmin -> cumulative degree days ---
cpc_dir = "/glade/derecho/scratch/yingxiao/FROM_CHEYENNE/CPC/"
tmax_files = glob.glob(os.path.join(cpc_dir, "*tmax*"))
tmin_files = glob.glob(os.path.join(cpc_dir, "*tmin*"))

ds_tmax = xr.open_mfdataset(tmax_files, combine="nested", compat="override",
                             concat_dim="time", coords={"time": "time"}).sortby("time")
ds_tmin = xr.open_mfdataset(tmin_files, combine="nested", compat="override",
                             concat_dim="time", coords={"time": "time"}).sortby("time")
ds_tmax["time"] = pd.to_datetime(ds_tmax["time"])

avg_temperature = (ds_tmax["tmax"] + ds_tmin["tmin"]) / 2
base_temperature = 5  # degC, adjust as needed
degree_days = xr.where(avg_temperature > base_temperature, avg_temperature - base_temperature, 0)
cumulative_degree_days = degree_days.groupby("time.year").cumsum(dim="time")

cdd_lat = cumulative_degree_days.coords["lat"].values
cdd_lon = cumulative_degree_days.coords["lon"].values
cdd_lon_grid, cdd_lat_grid = np.meshgrid(cdd_lon, cdd_lat)
cdd_points = np.column_stack((cdd_lat_grid.ravel(), cdd_lon_grid.ravel()))


In [ ]:
# --- CLM45 land-cover fractions + day length ---
ds_landcover = xr.open_dataset("/glade/derecho/scratch/yingxiao/Land-cover/CLM45_surface_emisvars.nc")
ds_grid = xr.open_dataset("/glade/derecho/scratch/yingxiao/Land-cover/rcm_lat_lon.nc")

# NOTE: adjust these variable names to match your actual CLM45 file contents
DBL = ds_landcover["DBL"]   # day length
ENL = ds_landcover["ENL"]   # evergreen needleleaf fraction
C3  = ds_landcover["C3"]    # C3 grass fraction
C4  = ds_landcover["C4"]    # C4 grass fraction
LC_lat = ds_grid["lat"].values
LC_lon = ds_grid["lon"].values


In [ ]:
def nearest_grid_value(lat, lon, grid_lat, grid_lon, data_array):
    """Find the nearest grid cell to (lat, lon) and return the corresponding data value.
    Generalizes the repeated per-city cdist/argmin blocks from the original notebook
    into a single reusable function."""
    distances = np.sqrt((grid_lat - lat) ** 2 + (grid_lon - lon) ** 2)
    closest_index = np.unravel_index(np.argmin(distances), distances.shape)
    return data_array[closest_index[0], closest_index[1]].values


def nearest_cdd_series(lat, lon):
    """Return the cumulative-degree-day time series nearest to (lat, lon)."""

    lon_0_360 = lon + 360
    distances = cdist([[lat, lon_0_360]], cdd_points)
    closest_index = np.argmin(distances)
    lat_idx = closest_index // len(cdd_lon)
    lon_idx = closest_index % len(cdd_lon)
    return cumulative_degree_days[:, lat_idx, lon_idx]


# Look up static land-cover / day-length covariates for every site once
site_covariates = {}
for code_, meta in STATIONS.items():
    site_covariates[code_] = {
        "DBL": nearest_grid_value(meta["lat"], meta["lon"], LC_lat, LC_lon, DBL),
        "ENL": nearest_grid_value(meta["lat"], meta["lon"], LC_lat, LC_lon, ENL),
        "C3":  nearest_grid_value(meta["lat"], meta["lon"], LC_lat, LC_lon, C3),
        "C4":  nearest_grid_value(meta["lat"], meta["lon"], LC_lat, LC_lon, C4),
        "CDD_series": nearest_cdd_series(meta["lat"], meta["lon"]),
    }


## 6. Attach covariates and build per-site predictor tables

For each site, append (Month, Day) columns pulled from the target array's date fields, plus the
CDD/day-length/land-cover covariates, onto the lidar-optical-property predictor array 

In [ ]:
def attach_covariates(X_predictors, y_target, month_idx, day_idx, cdd_series, dbl, enl, c3, c4):
    """Append Month, Day, CDD, DBL, ENL, C3, C4 as extra predictor columns.

    X_predictors : (n, m) array of lidar optical properties (Depo, Back, Extinc, Aod, ...)
    y_target     : (n, k) array used only to pull Month/Day columns for alignment
    """
    n = X_predictors.shape[0]
    months = y_target[:, month_idx]
    days = y_target[:, day_idx]
    dbl_col = np.ones(n) * dbl
    enl_col = np.ones(n) * enl
    c3_col = np.ones(n) * c3
    c4_col = np.ones(n) * c4
    # cdd_series must already be subset/aligned to this site's time steps before calling this
    return np.column_stack([months, days, X_predictors, cdd_series, dbl_col, enl_col, c3_col, c4_col])


# Column order after attach_covariates, used for feature-importance labeling later
PREDICTOR_COLUMNS = ["Month", "Day", "Depo", "Back", "Extinc", "Aod", "CDD", "DBL", "ENL", "C3", "C4"]

month_idx = y_columns.index("Month")
day_idx = y_columns.index("Day")

X_by_site = {}
y_by_site = {}
for code_, data in site_data.items():
    cov = site_covariates[code_]
    # --- CALIPSO-collocated records ---
    # TODO: subset cov["CDD_series"] to match the dates of this site's CALIPSO records
    # (original notebook does this via np.isin() date matching -- reproduce as needed)
    X_by_site[code_] = attach_covariates(
        data["X_cal"], data["y_cal"], month_idx, day_idx,
        cov["CDD_series"], cov["DBL"], cov["ENL"], cov["C3"], cov["C4"],
    )
    y_by_site[code_] = data["y_cal"]

    # --- MPLNET-collocated records (Silver Spring only) ---
    if "X_mpl" in data:
        X_by_site[f"{code_}_mpl"] = attach_covariates(
            data["X_mpl"], data["y_mpl"], month_idx, day_idx,
            cov["CDD_series"], cov["DBL"], cov["ENL"], cov["C3"], cov["C4"],
        )
        y_by_site[f"{code_}_mpl"] = data["y_mpl"]


## 7. Combine all sites into a single training table

`X_all` = predictors (lidar optical properties + Month/Day + meteorological/land-cover
covariates). `y_all` = pollen counts (target). 

In [ ]:
X_all = np.vstack(list(X_by_site.values()))
y_all = np.concatenate(list(y_by_site.values()))

print("X_all (predictors) shape:", X_all.shape)
print("y_all (pollen target) shape:", y_all.shape)


## 8. Train / test split

Standard scikit-learn convention: `train_test_split(X, y, ...)` returns
`X_train, X_test, y_train, y_test`

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=42
)

print("Training predictors shape:", X_train.shape)
print("Training target shape:", y_train.shape)
print("Testing predictors shape:", X_test.shape)
print("Testing target shape:", y_test.shape)


## 9. (Optional) Sample weighting

up-weighted samples in the top 80th percentile of certain predictor
columns, to help the model learn extreme/peak pollen events better. This is an optional.

In [ ]:
def get_highest_80_percent_indices(df, value_col="Value", group_col="Year"):
    """Return row indices in the top 20% of `value_col`, computed within each `group_col` group."""
    indices_to_update = []
    for _, group in df.groupby(group_col):
        threshold = np.percentile(group[value_col], 80)
        indices_to_update.extend(group[group[value_col] >= threshold].index.tolist())
    return indices_to_update


# Example: upweight extreme values, then renormalize -- adapt column indices as needed
sample_weights = np.ones(len(X_train))
# for i in range(X_train.shape[1]):
#     df_vals = pd.DataFrame({"Year": ..., "Value": X_train[:, i]})
#     idx = get_highest_80_percent_indices(df_vals)
#     sample_weights[idx] *= 1.2


## 10. Train the Random Forest model

In [ ]:
rf = RandomForestRegressor(
    n_estimators=1200,
    random_state=42,
    min_samples_split=16,
    min_samples_leaf=10,
    max_features="sqrt",
    max_depth=35,
    bootstrap=True,
)

rf.fit(X_train, y_train, sample_weight=sample_weights)


## 11. Evaluate 

In [ ]:
test_r2 = rf.score(X_test, y_test)
print("Test R^2:", test_r2)

predictions = rf.predict(X_test)
errors = np.abs(predictions - y_test)
print("Mean Absolute Error:", np.round(np.mean(errors), 2))


In [ ]:
# 5-fold cross-validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X_train, y_train, cv=kfold)
print("Mean CV R^2:", cv_scores.mean())


## 12. Feature importance

In [ ]:
importances = list(rf.feature_importances_)
feature_importances = sorted(
    zip(PREDICTOR_COLUMNS, importances), key=lambda x: x[1], reverse=True
)
for name, imp in feature_importances:
    print(f"Variable: {name:12s} Importance: {imp:.3f}")

fig = plt.figure(figsize=(10, 6))
names, vals = zip(*feature_importances)
plt.bar(names, vals)
plt.xticks(rotation=45)
plt.ylabel("Importance")
plt.title("Random Forest Feature Importances (predictors of pollen concentration)")
plt.tight_layout()
plt.show()


## 13. Per-species performance metrics

Compute RMSE, R², MAE, and SMAPE for each pollen species (each column of `y_test` /
`predictions`).

In [ ]:
species_columns = [c for c in y_columns if c not in ("Month", "Day", "Year", "Total")]
species_idx = [y_columns.index(c) for c in species_columns]

num_species = len(species_idx)
rmse_vals = np.zeros(num_species)
r2_vals = np.zeros(num_species)
mae_vals = np.zeros(num_species)
smape_vals = np.zeros(num_species)

for i, col in enumerate(species_idx):
    actual = y_test[:, col]
    pred = predictions[:, col]
    rmse_vals[i] = np.sqrt(mean_squared_error(actual, pred))
    r2_vals[i] = r2_score(actual, pred)
    mae_vals[i] = np.mean(np.abs(pred - actual))
    smape_vals[i] = np.sum(np.abs(pred - actual)) / np.sum(pred + actual)

metrics_df = pd.DataFrame({
    "Species": species_columns,
    "R2": r2_vals,
    "MAE": mae_vals,
    "SMAPE": smape_vals,
    "RMSE": rmse_vals,
})
metrics_df


In [ ]:
metrics_df.to_excel("pollen_model_metrics.xlsx", index=False)
